# Tertiary Demand Forecasting — Training

Model: **Polynomial Ridge + CatBoost** blend (VotingRegressor, weights 0.8 / 0.2), **no lag features**.
Target `tertiary` is log1p-transformed. Tertiary is driven almost directly by `activating_outlet` (corr 0.985), so concurrent features alone reach ~98%.
Saves `../models/tertiary_model.joblib` (loaded by `main_2.py`).


In [ ]:
import pandas as pd, numpy as np, math, joblib
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings; warnings.filterwarnings('ignore')


### 1. Features (concurrent only, no lags)


In [ ]:
TARGET='tertiary'
FEATS=['ret_stock','ret_dos','wod','activating_outlet','dbr_stock','dbr_dos','stocking_outlet','plc_factor',
       'seasonality_1(sin)','seasonality_2(cos)','total_stock','activation_ratio']
LOGF=['ret_stock','ret_dos','wod','activating_outlet','dbr_stock','dbr_dos','stocking_outlet','plc_factor','total_stock']

def seasonality(df):
    o=df.copy(); a=2*math.pi*pd.to_numeric(o['week'],errors='coerce').fillna(0)/59.0
    o['seasonality_1(sin)']=np.sin(a); o['seasonality_2(cos)']=np.cos(a); return o

def engineer(df):
    o=df.copy()
    o['total_stock']=pd.to_numeric(o['ret_stock'],errors='coerce').fillna(0)+pd.to_numeric(o['dbr_stock'],errors='coerce').fillna(0)
    o['activation_ratio']=pd.to_numeric(o['activating_outlet'],errors='coerce').fillna(0)/(pd.to_numeric(o['stocking_outlet'],errors='coerce').fillna(0)+1e-5)
    return o

def build_X(df):
    o=df.copy()
    for c in FEATS:
        if c not in o: o[c]=0.0
    X=o[FEATS].apply(pd.to_numeric,errors='coerce').fillna(0)
    for c in LOGF:
        if c in X: X[c]=np.log1p(np.clip(X[c],0,None))
    return X.replace([np.inf,-np.inf],np.nan).fillna(0)


### 2. Train on cleaned_tertiary.csv


In [ ]:
csv=pd.read_csv('cleaned_tertiary.csv')
csv=csv[csv[TARGET]>=0].copy()
csv=engineer(seasonality(csv))
X=build_X(csv); y=np.log1p(np.clip(csv[TARGET].values,0,None))

ridge_poly=make_pipeline(PolynomialFeatures(degree=2,include_bias=False), Ridge(alpha=10.0,random_state=42))
cat=CatBoostRegressor(iterations=800,learning_rate=0.05,depth=5,loss_function='MAE',random_seed=42,verbose=0,allow_writing_files=False)
model=VotingRegressor([('ridgepoly',ridge_poly),('cat',cat)], weights=[0.8,0.2])
model.fit(X,y)
print('Trained on', len(csv), 'rows')


### 3. Evaluate on the HEROSHAK test file


In [ ]:
def acc(a,b):
    a=np.asarray(a); b=np.clip(np.asarray(b),0,None)
    return max(0,100-np.mean(np.abs((a-b)/np.clip(np.abs(a),1,None))*100))

te=engineer(seasonality(pd.read_excel('HEROSHAK2025H11-Testing Upload.xlsx')))
pred=np.clip(np.expm1(model.predict(build_X(te))),0,None); yt=te[TARGET].values; wk=te['week'].values
m=wk!=1
print(f'All weeks : {acc(yt,pred):.2f}%  R2={r2_score(yt,pred):.3f}')
print(f'Excl wk1  : {acc(yt[m],pred[m]):.2f}%')
display(pd.DataFrame({'Week':wk,'Actual':yt,'Pred':np.round(pred,0),'APE%':np.round(np.abs(yt-pred)/np.clip(yt,1,None)*100,1)}))


### 4. Save model artifact


In [ ]:
import os; os.makedirs('../models',exist_ok=True)
joblib.dump({'model':model,'features':FEATS,'log_feature_cols':LOGF,'target':'tertiary',
             'target_log1p':True,'scaler':None,'model_version':'Tertiary_PolyRidge_CatBoost_v1'},
            '../models/tertiary_model.joblib')
print('Saved ../models/tertiary_model.joblib')
